# 🚀 Colab GGUF Model Runner
Run open-source GGUF language models inside Google Colab with GPU acceleration and an OpenAI-compatible FastAPI server.

In [ ]:
# @title 1. Environment & GPU Setup
import os
import shutil
import subprocess
import sys

print("🔍 Checking GPU availability...")
nvidia_smi = shutil.which("nvidia-smi")
GPU_AVAILABLE = False
if nvidia_smi is None:
    print("⚠️ NVIDIA tools are unavailable. Continuing in CPU mode. For GPU acceleration, select a GPU runtime and rerun this cell.")
else:
    gpu_status = subprocess.run([nvidia_smi], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if gpu_status.returncode != 0:
        print("⚠️ No NVIDIA GPU detected. Continuing in CPU mode.")
    else:
        GPU_AVAILABLE = True
        print("✅ GPU detected successfully:")
        for line in gpu_status.stdout.splitlines()[:8]:
            print("   " + line)

print("\n📦 Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "nest_asyncio", "pyngrok", "requests", "pydantic", "tqdm"])
if GPU_AVAILABLE:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "llama-cpp-python", "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121", "--no-cache-dir"])
    except subprocess.CalledProcessError:
        print("⚠️ CUDA wheel installation failed; compiling llama-cpp-python with CUDA support.")
        env = os.environ.copy()
        env["CMAKE_ARGS"] = "-DGGML_CUDA=on"
        subprocess.check_call([sys.executable, "-m", "pip", "install", "llama-cpp-python", "--no-cache-dir"], env=env)
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python", "--no-cache-dir"])
print("✅ All dependencies installed successfully!")

In [ ]:
# @title 2. Data Input & Configuration Form
# @markdown Specify the model download link and runtime options.
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct-GGUF/resolve/main/qwen2.5-coder-14b-instruct-q4_k_m.gguf"  # @param {type:"string"}
USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_FOLDER_NAME = "colab_models"  # @param {type:"string"}
NGROK_AUTHTOKEN = ""  # @param {type:"string"}
N_GPU_LAYERS = -1  # @param {type:"integer"}
CONTEXT_SIZE = 4096  # @param {type:"integer"}

from pathlib import Path
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        TARGET_DIR = Path('/content/drive/MyDrive') / DRIVE_FOLDER_NAME
    except Exception as e:
        print(f"⚠️ Failed to mount Google Drive ({e}). Falling back to local storage.")
        TARGET_DIR = Path('/content/models')
else:
    TARGET_DIR = Path('/content/models')
TARGET_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Storage set to: {TARGET_DIR}")
raw_filename = MODEL_URL.split('?')[0].rstrip('/').split('/')[-1]
if not raw_filename.endswith('.gguf'):
    raw_filename = 'model.gguf'
LOCAL_MODEL_PATH = TARGET_DIR / raw_filename
print(f"Target model file: {LOCAL_MODEL_PATH}")

In [ ]:
# @title 3. Download Model & Verify Integrity
import requests
from tqdm.auto import tqdm
def download_file_with_progress(url: str, destination: Path):
    if destination.exists() and destination.stat().st_size > 10 * 1024 * 1024:
        print(f"⚡ Model already exists at: {destination} ({destination.stat().st_size / (1024**3):.2f} GB). Skipping download.")
        return
    print(f"📥 Downloading model from: {url}")
    with requests.get(url, stream=True, headers={"User-Agent": "ColabModelRunner/1.0"}, timeout=60) as response:
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))
        with open(destination, 'wb') as file, tqdm(desc=destination.name, total=total_size, unit='iB', unit_scale=True, unit_divisor=1024) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
                    bar.update(len(chunk))
    print(f"\n✅ Download completed: {destination.name} ({destination.stat().st_size / (1024 ** 3):.2f} GB)")
download_file_with_progress(MODEL_URL, LOCAL_MODEL_PATH)
assert LOCAL_MODEL_PATH.exists() and LOCAL_MODEL_PATH.stat().st_size > 1024 * 1024, 'Model file is missing or corrupted!'
print('✅ Model integrity verified.')

In [ ]:
# @title 4. FastAPI Server & OpenAI-Compatible Endpoint with Tunneling
import re
import secrets
import subprocess
import threading
import time
import nest_asyncio
import uvicorn
from fastapi import Depends, FastAPI, HTTPException, Security
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel
from typing import List, Optional
from llama_cpp import Llama

nest_asyncio.apply()
API_KEY = f'sk-colab-{secrets.token_hex(16)}'
security_scheme = HTTPBearer()

def verify_api_key(credentials: HTTPAuthorizationCredentials = Security(security_scheme)):
    if credentials.scheme.lower() != 'bearer' or credentials.credentials != API_KEY:
        raise HTTPException(status_code=401, detail='Invalid API Key')
    return credentials.credentials

print(f'🔄 Loading GGUF model: {LOCAL_MODEL_PATH.name}...')
llm = Llama(model_path=str(LOCAL_MODEL_PATH), n_gpu_layers=N_GPU_LAYERS if GPU_AVAILABLE else 0, n_ctx=CONTEXT_SIZE, verbose=False)
print('✅ Model loaded successfully!')

app = FastAPI(title='Colab Model Runner')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = 'default'
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.9
    max_tokens: Optional[int] = 512

@app.get('/')
def root():
    return {'status': 'online', 'model': LOCAL_MODEL_PATH.name}

@app.post('/v1/chat/completions', dependencies=[Depends(verify_api_key)])
def chat_completions(req: ChatCompletionRequest):
    try:
        formatted_messages = [{'role': m.role, 'content': m.content} for m in req.messages]
        return llm.create_chat_completion(messages=formatted_messages, temperature=req.temperature, top_p=req.top_p, max_tokens=req.max_tokens)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

public_url = None
if NGROK_AUTHTOKEN.strip():
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    public_url = ngrok.connect(8000, 'http').public_url
else:
    try:
        proc = subprocess.Popen(['npx', '--yes', 'localtunnel', '--port', '8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        deadline = time.time() + 20
        while time.time() < deadline and proc.poll() is None:
            line = proc.stdout.readline()
            match = re.search(r'https://[^\s]+', line)
            if match:
                public_url = match.group(0).rstrip('.')
                break
        if not public_url:
            raise RuntimeError('localtunnel did not return a public HTTPS URL')
    except Exception as e:
        print(f'Tunnel warning: {e}')

if not public_url:
    raise RuntimeError('No public tunnel URL is available. Add an ngrok token or retry localtunnel.')
base_endpoint = public_url.replace('http://', 'https://').rstrip('/')

print('\n=======================================================')
print('🎉 COLAB MODEL RUNNER IS READY!')
print('=======================================================')
print(f'🌐 Base URL   : {base_endpoint}')
print(f'🔑 API Key    : {API_KEY}')
print(f'⚡ Endpoint   : {base_endpoint}/v1/chat/completions')
print('=======================================================')

curl_example = f'''curl -X POST "{base_endpoint}/v1/chat/completions" \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer {API_KEY}" \
  -d '{{
    "messages": [
      {{"role": "user", "content": "Hello!"}}
    ],
    "temperature": 0.7,
    "max_tokens": 120
  }}'''
print(curl_example)
print('=======================================================')
print('Keep this Colab tab open to maintain the active server.')